# AutoResearch on Colab
基於 [karpathy/autoresearch](https://github.com/karpathy/autoresearch)

**使用方式：**
1. Runtime → Change runtime type → 選 **GPU (T4+)**
2. 從上到下依序執行每個 cell
3. 訓練完成後結果會自動回傳到 MD.Piece 後端

**功能：**
- 單次訓練 baseline 或自訂實驗
- 自動實驗循環（多次迭代）
- 結果自動回傳 + results.tsv 批次匯入

In [ ]:
# Step 1: 確認 GPU
!nvidia-smi
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU! Go to Runtime → Change runtime type → GPU")

In [ ]:
# Step 2: 安裝 uv
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] = os.path.expanduser('~/.local/bin') + ':' + os.environ['PATH']

In [ ]:
# Step 3: Clone autoresearch
!git clone https://github.com/karpathy/autoresearch.git
%cd autoresearch

In [ ]:
# Step 4: 安裝依賴
!uv sync

In [ ]:
# Step 5: 資料準備（下載 + tokenizer，約 2 分鐘）
!uv run prepare.py --num-shards 10

In [ ]:
# Step 6: 訓練（約 5 分鐘）
import time
import subprocess
import re

def run_experiment(name="baseline"):
    """執行一次訓練並回傳結果 dict"""
    start = time.time()
    result = subprocess.run(
        ["uv", "run", "train.py"],
        capture_output=True, text=True
    )
    duration = time.time() - start

    print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr[-500:])

    val_bpb = None
    train_loss = None
    steps = None

    for line in result.stdout.split("\n"):
        m = re.search(r'val_bpb[=:\s]+(\d+\.\d+)', line)
        if m:
            val_bpb = float(m.group(1))
        m = re.search(r'(?:train_)?loss[=:\s]+(\d+\.\d+)', line)
        if m:
            train_loss = float(m.group(1))
        m = re.search(r'step[=:\s]+(\d+)', line)
        if m:
            steps = int(m.group(1))

    print(f"\n--- 結果 ---")
    print(f"val_bpb: {val_bpb}")
    print(f"train_loss: {train_loss}")
    print(f"steps: {steps}")
    print(f"duration: {duration:.0f}s")

    return {
        "name": name,
        "val_bpb": val_bpb,
        "train_loss": train_loss,
        "steps": steps,
        "duration_seconds": duration,
        "returncode": result.returncode,
    }

# 執行 baseline
baseline = run_experiment("colab-baseline")

In [ ]:
# Step 7: 回傳結果到 MD.Piece 後端
# ⚠️ 如果後端跑在本機，請用 ngrok 或改成你的公開 URL
MDPIECE_API = "http://localhost:8000"  # ← 改成你的後端 URL

import requests
import json

def submit_to_mdpiece(exp_result, kept=None, notes_extra=""):
    """提交實驗結果到 MD.Piece API"""
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "unknown"
    payload = {
        "name": exp_result["name"],
        "val_bpb": exp_result["val_bpb"],
        "train_loss": exp_result["train_loss"],
        "steps": exp_result["steps"],
        "duration_seconds": exp_result["duration_seconds"],
        "notes": f"Colab {gpu_name}, PyTorch {torch.__version__}" + (f" | {notes_extra}" if notes_extra else ""),
        "model_config_summary": "default autoresearch config",
        "kept": kept,
    }
    try:
        resp = requests.post(f"{MDPIECE_API}/research/", json=payload, timeout=10)
        print(f"結果已回傳: {resp.json().get('message', '')}")
        return True
    except Exception as e:
        print(f"無法回傳到 MD.Piece 後端: {e}")
        print(f"手動提交資料:\n{json.dumps(payload, indent=2, ensure_ascii=False)}")
        return False

submit_to_mdpiece(baseline, kept=True)

In [ ]:
# Step 8（選用）: 自動實驗循環
# 設定要跑幾輪實驗（每輪約 5 分鐘）
NUM_ITERATIONS = 3  # ← 改成你要的輪數

import datetime
import csv

best_bpb = baseline["val_bpb"]
results = [baseline]

# 超參數搜尋空間（示範）
import random
HYPERPARAM_VARIANTS = [
    ("DEPTH", [6, 8, 10, 12]),
    ("ASPECT_RATIO", [48, 64, 80]),
    ("TOTAL_BATCH_SIZE", [2**18, 2**19, 2**20]),
]

for i in range(NUM_ITERATIONS):
    # 隨機選一個超參數修改
    param_name, values = random.choice(HYPERPARAM_VARIANTS)
    new_val = random.choice(values)
    exp_name = f"exp-{i+1:03d}-{param_name}={new_val}"

    print(f"\n{'='*60}")
    print(f"實驗 {i+1}/{NUM_ITERATIONS}: {exp_name}")
    print(f"{'='*60}")

    # 讀取 train.py 並修改超參數
    with open("train.py", "r") as f:
        code = f.read()
    original_code = code

    import re as _re
    pattern = _re.compile(rf'^({param_name}\s*=\s*)(.+)$', _re.MULTILINE)
    if pattern.search(code):
        code = pattern.sub(rf'\g<1>{new_val}', code)
        with open("train.py", "w") as f:
            f.write(code)
        print(f"修改 {param_name} = {new_val}")
    else:
        print(f"找不到 {param_name}，跳過")
        continue

    # 執行訓練
    exp = run_experiment(exp_name)

    # 評估結果
    kept = False
    if exp["returncode"] == 0 and exp["val_bpb"] is not None:
        if best_bpb is None or exp["val_bpb"] < best_bpb:
            print(f"改善！{best_bpb} → {exp['val_bpb']}")
            best_bpb = exp["val_bpb"]
            kept = True
        else:
            print(f"未改善（{exp['val_bpb']} >= {best_bpb}），還原")
            with open("train.py", "w") as f:
                f.write(original_code)
    else:
        print("訓練失敗或無結果，還原")
        with open("train.py", "w") as f:
            f.write(original_code)

    exp["kept"] = kept
    results.append(exp)

    # 回傳到 MD.Piece
    submit_to_mdpiece(exp, kept=kept, notes_extra=f"{param_name}={new_val}")

# 寫入 results.tsv
with open("results.tsv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["name", "val_bpb", "train_loss", "steps", "duration", "kept", "timestamp"], delimiter="\t")
    writer.writeheader()
    for r in results:
        writer.writerow({
            "name": r["name"],
            "val_bpb": r.get("val_bpb", ""),
            "train_loss": r.get("train_loss", ""),
            "steps": r.get("steps", ""),
            "duration": round(r.get("duration_seconds", 0)),
            "kept": r.get("kept", ""),
            "timestamp": datetime.datetime.utcnow().isoformat(),
        })

print(f"\n最佳 val_bpb: {best_bpb}")
print(f"結果已存入 results.tsv（{len(results)} 筆）")

In [ ]:
# Step 9（選用）: 批次匯入 results.tsv 到 MD.Piece
import os
if os.path.exists("results.tsv"):
    try:
        with open("results.tsv", "rb") as f:
            resp = requests.post(
                f"{MDPIECE_API}/research/batch",
                files={"file": ("results.tsv", f, "text/tab-separated-values")},
                timeout=30,
            )
        print(f"批次匯入: {resp.json()}")
    except Exception as e:
        print(f"批次匯入失敗: {e}")
        print("請手動下載 results.tsv 並在 MD.Piece 前端匯入")
else:
    print("results.tsv 不存在，請先執行 Step 8")